# Framing Annotation — Step 2: Improve Coverage of Draft Annotation Instructions

- **Step 2.1** — Test draft annotation instructions on a small random sample and inspect results  
- **Step 2.2** — Run multiple annotation runs (briseus equivalent) to identify hard cases where the model disagrees with itself

The goal at this stage is **not** to achieve high reliability yet — it is to make sure your annotation instructions cover all cases you encounter in the data. Hard cases and mislabeled cases will tell you where to add decision rules and clarify your codebook.

## Setup

Load libraries and configure the API connection. The API key is loaded from a `.env` file — make sure you have a `.env` file in your project root with:

```
CAC_API_KEY=your_key_here
```

In [8]:
import pandas as pd
import requests
import os
import time
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

print(f"API key loaded: {'YES' if api_key else 'NO — check your .env file'}")
print(f"Host: {HOST}")
print(f"Model: {MODEL}")

API key loaded: YES
Host: https://maki.uni-mannheim.de/v1
Model: ministral-3-14b


## Step 1 Draft: Annotation Instructions

These are the **Step 1 annotation instructions** written based on the theoretical framework from Freudenthaler et al. (under review). The **Immigration and Crime Frame** (Security Threat Frame in the paper) belongs to dimension 5 of the framework — the ideological layer that links structural inequalities to attitudes and behaviors.

According to the paper, security threat frames portray racialized groups as implicitly or explicitly causing crime, justifying harsher policing and restrictive immigration policies.

> **Note:** These instructions are a first draft and will be refined iteratively through Steps 2.1 and 2.2. Modify the `instructions` and `reminder` variables below after each round of inspection.

In [9]:
instructions = """Sie sind ein Experte für Inhaltsanalyse in einer Studie zur medialen Darstellung ethnischer und sozialer Gruppen in deutschen Nachrichten. Ihre Aufgabe ist es, zu klassifizieren, ob ein gegebener Nachrichtenabsatz einen Einwanderungs- und Kriminalitätsrahmen (Immigration and Crime Frame) enthält — das heißt, ob der Absatz eine Gruppe (bezeichnet als [Gruppe 1] oder [andere Gruppe]) explizit oder implizit mit Kriminalität, kriminellem Verhalten, Sicherheitsbedrohungen oder abweichendem Verhalten in Verbindung bringt.

Sie erhalten Mehrfachabsätze aus deutschen Zeitungsartikeln. Konzentrieren Sie sich ausschließlich auf das, was explizit im Text steht. Schließen Sie keine Gruppenassoziationen, die nicht direkt ausgedrückt werden.

CRIME_FRAME: Ein Absatz erhält dieses Label, wenn er [Gruppe 1] oder [andere Gruppe] explizit oder implizit mit Kriminalität, kriminellem Verhalten, Sicherheitsbedrohungen oder abweichendem Verhalten in Verbindung bringt. Dies umfasst:
- Direkte Assoziation: Die Gruppe wird als Täter, Verdächtiger oder Planer einer Straftat beschrieben
- Implizite Assoziation: Die Gruppe wird im Kontext von Kriminalstatistiken, Ermittlungen oder Polizeimaßnahmen erwähnt, bei denen die Gruppenzugehörigkeit hervorgehoben wird
- Sicherheitsbedrohung: Die Gruppe wird als Gefahr für die öffentliche Sicherheit oder nationale Sicherheit dargestellt
- Kausale Zuschreibung: Die Anwesenheit der Gruppe wird als Ursache für steigende Kriminalität dargestellt
- Politische Rahmung: Restriktive Maßnahmen werden als Reaktion auf der Gruppe zugeschriebene Kriminalität gerechtfertigt

NO_CRIME_FRAME: Ein Absatz erhält dieses Label, wenn:
- Die Gruppe gar nicht erwähnt wird
- Die Gruppe erwähnt wird, aber nicht im Zusammenhang mit Kriminalität oder Sicherheitsbedrohungen
- Die Gruppe als Opfer von Straftaten dargestellt wird, nicht als Täter
- Kriminalität auf soziale Bedingungen zurückgeführt wird, nicht auf die Gruppe selbst
- Der Absatz den Kriminalitätsrahmen explizit in Frage stellt oder widerlegt

Entscheidungsregeln:
- Wenn [Gruppe 1] oder [andere Gruppe] das OPFER einer Straftat ist, vergeben Sie NO_CRIME_FRAME
- Wenn Kriminalität auf Armut oder soziale Ausgrenzung zurückgeführt wird, vergeben Sie NO_CRIME_FRAME
- Wenn der Absatz einen Kriminalitätsrahmen anficht oder widerlegt, vergeben Sie NO_CRIME_FRAME
- Im Zweifelsfall fragen Sie: Würde ein Leser vernünftigerweise schlussfolgern, dass die Gruppe mit Kriminalität in Verbindung gebracht wird? Wenn ja: CRIME_FRAME. Wenn nein: NO_CRIME_FRAME"""

reminder = "Klassifizieren Sie, ob der Absatz [Gruppe 1] oder [andere Gruppe] mit Kriminalität, Sicherheitsbedrohungen oder abweichendem Verhalten in Verbindung bringt. Vergeben Sie CRIME_FRAME nur, wenn die Gruppe als bedrohlich oder abweichend dargestellt wird — nicht wenn sie als Opfer erscheint oder Kriminalität auf strukturelle Ursachen zurückgeführt wird."

print("Instructions and reminder loaded.")

Instructions and reminder loaded.


## Core Functions

Three functions power the annotation pipeline:

- `annotate()` — sends a single paragraph to the API and returns the raw model output
- `parse_label()` — extracts the label from the raw output, handling misspellings and drift
- `parse_explanation()` — extracts the explanation sentence from the raw output

**On label drift:** Small LLMs sometimes output unexpected labels (e.g. `CRIME` instead of `CRIME_FRAME`, or a misspelling). The `parse_label()` function handles this by doing a fuzzy match rather than an exact match. If more than 5% of your outputs are `UNCLEAR`, revisit the reminder and output format instructions.

In [10]:
def annotate(text, instructions, reminder, temperature=0.0):
    """Send a paragraph to the API for annotation."""
    prompt = f"{instructions}\n\nNow annotate this paragraph:\n\n{text}\n\n{reminder}\n\nRespond in this format exactly:\nLabel: <CRIME_FRAME or NO_CRIME_FRAME>\nExplanation: <one sentence explaining your choice>"

    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": [
            {"role": "system", "content": "Sie sind ein Experte für Inhaltsanalyse. Befolgen Sie die Annotationsanweisungen genau und antworten Sie immer im angegebenen Format."},
            {"role": "user", "content": prompt}
        ]
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    for attempt in range(3):
        try:
            response = requests.post(
                f"{HOST}/chat/completions",
                json=payload,
                headers=headers,
                timeout=120
            )
            if response.status_code == 200:
                return response.json()['choices'][0]['message']['content'].strip()
            else:
                print(f"  [attempt {attempt+1}] Status {response.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] Error: {e}")
            time.sleep(5)
    return "ERROR"


def parse_label(raw_output):
    """Extract and normalize label from raw model output.
    Handles misspellings and label drift from small LLMs.
    """
    raw_upper = raw_output.upper()
    # Check NO_CRIME_FRAME first (it contains CRIME_FRAME as substring)
    if "NO_CRIME_FRAME" in raw_upper or "NO CRIME FRAME" in raw_upper:
        return "NO_CRIME_FRAME"
    elif "CRIME_FRAME" in raw_upper or "CRIME FRAME" in raw_upper:
        return "CRIME_FRAME"
    else:
        return "UNCLEAR"  # label drift — needs manual inspection


def parse_explanation(raw_output):
    """Extract explanation sentence from raw model output."""
    for line in raw_output.split("\n"):
        if line.lower().startswith("explanation:"):
            return line[len("explanation:"):].strip()
    return raw_output  # fallback: return full output


print("Functions loaded.")

Functions loaded.


## Step 2.1 — Test Draft Annotation Instructions

We run the annotation on a **random sample of 50 rows** from the translated dataset. This is equivalent to the `bacchuss()` call in the R vignette.

After running, visually inspect the results and note:
- Which cases were correctly labelled?
- Are there clear cases not described in your coding instructions?
- What are general problems with the instructions?

Then go back and update the `instructions` and `reminder` in the cell above.

In [11]:
# ── Config ───────────────────────────────────────────────────────────────
SAMPLE_SIZE_2_1 = 50    # number of rows to annotate in step 2.1
RANDOM_SEED     = 42    # for reproducibility — use the same seed each run
OUTPUT_2_1      = "results/annotation_step2_1.csv"
# ─────────────────────────────────────────────────────────────────────────

os.makedirs("results", exist_ok=True)

df = pd.read_csv("dataset/all_multi_paragraphs_2022_2026_translated.csv")
sample_2_1 = df.sample(n=SAMPLE_SIZE_2_1, random_state=RANDOM_SEED).copy().reset_index(drop=True)

print(f"Dataset loaded: {len(df)} rows total")
print(f"Sample size: {len(sample_2_1)} rows")
print(f"Running annotation at temperature=0.0 (deterministic)...\n")

Dataset loaded: 30000 rows total
Sample size: 50 rows
Running annotation at temperature=0.0 (deterministic)...



In [12]:
# Groups most likely to have crime frames based on your data
crime_relevant_groups = ["REFUG", "ASYL", "MIGR", "XKX", "SYR", "AFG"]

targeted_sample = df[df["group"].isin(crime_relevant_groups)].sample(
    n=50, random_state=42
).reset_index(drop=True)

print(targeted_sample["group"].value_counts())

group
REFUG    35
MIGR      8
ASYL      4
SYR       2
AFG       1
Name: count, dtype: int64


In [13]:
results_2_1 = []

for i, row in targeted_sample.iterrows():
    raw = annotate(str(row["text_block"]), instructions, reminder, temperature=0.0)
    label = parse_label(raw)
    explanation = parse_explanation(raw)

    results_2_1.append({
        "article_id":    row["article_id"],
        "par_index":     row["par_index"],
        "group":         row["group"],
        "text_block_en": row["text_block"],
        "raw_output":    raw,
        "label":         label,
        "explanation":   explanation
    })

    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/50] done — CRIME_FRAME so far: {sum(1 for r in results_2_1 if r['label'] == 'CRIME_FRAME')}")

annotation_2_1 = pd.DataFrame(results_2_1)
annotation_2_1.to_csv("results/annotation_2_1.csv", index=False)

print(f"\nDone!")
print(annotation_2_1["label"].value_counts())
print(f"\nCRIME_FRAME rate: {len(annotation_2_1[annotation_2_1['label']=='CRIME_FRAME'])/len(annotation_2_1)*100:.1f}%")

  [10/50] done — CRIME_FRAME so far: 0
  [20/50] done — CRIME_FRAME so far: 0
  [30/50] done — CRIME_FRAME so far: 1
  [40/50] done — CRIME_FRAME so far: 1
  [50/50] done — CRIME_FRAME so far: 1

Done!
label
NO_CRIME_FRAME    49
CRIME_FRAME        1
Name: count, dtype: int64

CRIME_FRAME rate: 2.0%


### Inspect Results

Check the label distribution and flag any `UNCLEAR` labels (label drift). If more than **5% of labels are UNCLEAR**, adjust the reminder and output format instructions and rerun.

Use the output below to visually inspect cases — especially CRIME_FRAME cases — and ask:
- Does the label match what you would expect?
- Are there edge cases your instructions don't cover yet?

In [14]:
# Label distribution
total = len(annotation_2_1)
print("=== Label Distribution ===")
for label, count in annotation_2_1["label"].value_counts().items():
    print(f"  {label}: {count} ({count/total*100:.1f}%)")

# Flag unclear labels
unclear = annotation_2_1[annotation_2_1["label"] == "UNCLEAR"]
if len(unclear) > 0:
    pct = len(unclear) / total * 100
    print(f"\nUNCLEAR labels: {len(unclear)} ({pct:.1f}%)")
    if pct > 5:
        print("   → More than 5% unclear. Consider adjusting the reminder and output format.")
    print(unclear[["text_block_en", "raw_output"]].to_string())
else:
    print("\n✓ No unclear labels.")

=== Label Distribution ===
  NO_CRIME_FRAME: 49 (98.0%)
  CRIME_FRAME: 1 (2.0%)

✓ No unclear labels.


In [15]:
# Browse all results — sort by label for easier inspection
annotation_2_1[["group", "label", "explanation", "text_block_en"]].sort_values("label")

,group,label,explanation,text_block_en
22,MIGR,CRIME_FRAME,Der Absatz beschreibt **[Gruppe 1]** explizit ...,"Gabriele Zull, parteilose Oberbürg..."
0,ASYL,NO_CRIME_FRAME,Der Absatz beschreibt [Gruppe 1] und [andere G...,"Die Betten sind noch nicht bezogen, Decken und..."
27,MIGR,NO_CRIME_FRAME,Der Absatz beschreibt ausschließlich die Auswa...,Um die JahrhundertwendeIm Mai 1886 heirateten ...
28,MIGR,NO_CRIME_FRAME,Der Absatz beschreibt die [Andere Gruppe] auss...,Hennrich: Die regulären Busse fuhren nicht meh...
29,REFUG,NO_CRIME_FRAME,Der Absatz thematisiert ausschließlich die **U...,Wie werden [Gruppe 1] verteilt?\n\nDas ist der...
30,ASYL,NO_CRIME_FRAME,Der Absatz erwähnt zwar die Anwesenheit von [G...,In Einsiedel kann die Stimmung schnell umschla...
31,AFG,NO_CRIME_FRAME,Der Absatz beschreibt **[Gruppe 1]** ausschlie...,Das Regime drängt [Gruppe 1] aus dem öffentlic...
32,REFUG,NO_CRIME_FRAME,Der Absatz erwähnt [Gruppe 1] und [andere Grup...,\n\nMAINZ. Die Stadt Mainz wird [Gruppe 1] kün...
33,REFUG,NO_CRIME_FRAME,Der Absatz thematisiert ausschließlich die hum...,ALZEY (red). Millionen Menschen sind seit Begi...
34,REFUG,NO_CRIME_FRAME,Der Absatz thematisiert die Gruppe ausschließl...,Aktuell suchen auch viele Menschen aus der Ukr...


In [16]:
# Show only CRIME_FRAME cases for focused inspection
crime_cases = annotation_2_1[annotation_2_1["label"] == "CRIME_FRAME"]
print(f"CRIME_FRAME cases: {len(crime_cases)}")
crime_cases[["group", "explanation", "text_block_en"]]

CRIME_FRAME cases: 1


,group,explanation,text_block_en
22,MIGR,Der Absatz beschreibt **[Gruppe 1]** explizit ...,"Gabriele Zull, parteilose Oberbürg..."


## Step 2.2 — Check for Hard Cases (briseus equivalent)

This step runs the annotation **multiple times with higher temperature** (0.7) on a subset of rows. Higher temperature introduces randomness — if the model consistently agrees with itself across runs, the case is clear. If it disagrees, the case is hard and the instructions need clarification.

This is the Python equivalent of the `briseus()` function from the `bacchuss` R package.

After running, sort by agreement (lowest first) and inspect the hard cases:
- Why is the model uncertain?
- Can you add a decision rule to resolve the ambiguity?
- Are these genuinely borderline cases, or a problem with the instructions?

> **Note:** This runs `n_runs × sample_size` API calls. With n_runs=5 and sample_size=20, that is 100 calls. Adjust accordingly.

In [17]:
# ── Config ───────────────────────────────────────────────────────────────
SAMPLE_SIZE_2_2 = 20    # number of rows for briseus (keep small — n_runs × this = total API calls)
N_RUNS          = 5     # number of annotation runs per paragraph
TEMPERATURE     = 0.7   # higher temperature = more variation between runs
OUTPUT_2_2      = "results/annotation_step2_2.csv"
# ─────────────────────────────────────────────────────────────────────────

# Draw a fresh random sample (different from 2.1 to cover more examples)
sample_2_2 = df.sample(n=SAMPLE_SIZE_2_2, random_state=99).copy().reset_index(drop=True)

print(f"Sample size: {len(sample_2_2)} rows")
print(f"Runs per paragraph: {N_RUNS}")
print(f"Total API calls: {len(sample_2_2) * N_RUNS}")
print(f"Temperature: {TEMPERATURE}")

Sample size: 20 rows
Runs per paragraph: 5
Total API calls: 100
Temperature: 0.7


In [18]:
results_2_2 = []

for i, row in sample_2_2.iterrows():
    text = str(row["text_block"])
    run_labels = []

    for run in range(N_RUNS):
        raw = annotate(text, instructions, reminder, temperature=TEMPERATURE)
        run_labels.append(parse_label(raw))

    # Majority vote as final label
    majority_label = max(set(run_labels), key=run_labels.count)
    agreement = run_labels.count(majority_label) / N_RUNS

    result = {
        "article_id":    row["article_id"],
        "par_index":     row["par_index"],
        "group":         row["group"],
        "text_block_en": text,
        "final_label":   majority_label,
        "agreement":     agreement,
    }
    for r, lbl in enumerate(run_labels):
        result[f"run_{r+1}"] = lbl

    results_2_2.append(result)
    print(f"  [{i+1}/{len(sample_2_2)}] {majority_label} (agreement: {agreement:.0%}) | runs: {run_labels}")

annotation_2_2 = pd.DataFrame(results_2_2).sort_values("agreement").reset_index(drop=True)
annotation_2_2.to_csv(OUTPUT_2_2, index=False)

print(f"\nBriseus complete! Saved to {OUTPUT_2_2}")

  [1/20] NO_CRIME_FRAME (agreement: 100%) | runs: ['NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME']
  [2/20] NO_CRIME_FRAME (agreement: 100%) | runs: ['NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME']
  [3/20] NO_CRIME_FRAME (agreement: 100%) | runs: ['NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME']
  [4/20] NO_CRIME_FRAME (agreement: 100%) | runs: ['NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME']
  [5/20] NO_CRIME_FRAME (agreement: 100%) | runs: ['NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME']
  [6/20] NO_CRIME_FRAME (agreement: 100%) | runs: ['NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME']
  [7/20] NO_CRIME_FRAME (agreement: 100%) | runs: ['NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME']
  [8/20] NO_C

### Inspect Hard Cases

Cases with **agreement < 80%** are hard cases — the model is uncertain and your instructions likely need a clearer decision rule for these.

For each hard case, ask:
1. What makes this paragraph ambiguous?
2. Can you write a decision rule that resolves the ambiguity?
3. Is this a genuine borderline case that needs a new label or sub-category?

Then go back to the **Annotation Instructions** cell above, update the instructions, and rerun both 2.1 and 2.2.

In [19]:
run_cols = [f"run_{r+1}" for r in range(N_RUNS)]

print("=== Full Results (sorted by agreement, lowest first) ===")
annotation_2_2[["agreement", "final_label", "text_block_en"] + run_cols]

=== Full Results (sorted by agreement, lowest first) ===


,agreement,final_label,text_block_en,run_1,run_2,run_3,run_4,run_5
0,1.0,NO_CRIME_FRAME,Er hatte recht. Zu den kuriosesten Regeln im L...,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME
1,1.0,NO_CRIME_FRAME,Keine Mannschaft bei diesem Turnier in Katar s...,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME
2,1.0,NO_CRIME_FRAME,Jeder siebte Einwohner ist aus dem Ausland\n\...,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME
3,1.0,NO_CRIME_FRAME,Angesichts der Schwere des Vorwurfs ließe sich...,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME
4,1.0,NO_CRIME_FRAME,\n\nUnd dann ist es auch schon wieder vorbei. ...,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME
5,1.0,NO_CRIME_FRAME,"Nur das Erdgeschoss besteht nicht aus Holz, so...",NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME
6,1.0,NO_CRIME_FRAME,Der 34-Jährige spielt bei Sporting Pelt in der...,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME
7,1.0,NO_CRIME_FRAME,\n\nMehr als ein amüsiertes Lächeln ernten Spo...,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME
8,1.0,NO_CRIME_FRAME,Schönberg: Jugendherberge soll Notunterkunft w...,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME
9,1.0,NO_CRIME_FRAME,"Die Stadt hat die Aufgaben, Unterkünfte bereit...",NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME


In [20]:
hard_cases = annotation_2_2[annotation_2_2["agreement"] < 0.8]
print(f"Hard cases (agreement < 80%): {len(hard_cases)} out of {len(annotation_2_2)}")

for _, row in hard_cases.iterrows():
    runs = [row[f"run_{r+1}"] for r in range(N_RUNS)]
    print(f"\n{'='*60}")
    print(f"Agreement: {row['agreement']:.0%} | Final label: {row['final_label']}")
    print(f"Runs: {runs}")
    print(f"Group: {row['group']}")
    print(f"Text: {str(row['text_block_en'])[:500]}...")
    print("→ TODO: Why is this hard? What decision rule would resolve it?")

Hard cases (agreement < 80%): 0 out of 20


In [21]:
# Agreement summary
print("=== Agreement Distribution ===")
print(annotation_2_2["agreement"].describe())
print(f"\nFull agreement (100%): {len(annotation_2_2[annotation_2_2['agreement'] == 1.0])} rows")
print(f"Hard cases (<80%):     {len(annotation_2_2[annotation_2_2['agreement'] < 0.8])} rows")
print(f"Very hard (<60%):      {len(annotation_2_2[annotation_2_2['agreement'] < 0.6])} rows")

=== Agreement Distribution ===
count    20.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
Name: agreement, dtype: float64

Full agreement (100%): 20 rows
Hard cases (<80%):     0 rows
Very hard (<60%):      0 rows


## Next Steps

After inspecting the results:

1. **Note problems** — write down what went wrong and why in a markdown cell or comment
2. **Update instructions** — go back to the instructions cell and add/clarify decision rules
3. **Repeat 2.1 and 2.2** — rerun both steps with the updated instructions
4. **Stop when** your instructions broadly cover the cases you see, even if reliability is not yet perfect

Once coverage is broad, move to **Step 3** — testing on a curated sample of hard and soft cases with human gold-standard labels, adding chain-of-thought prompting and few-shot examples.

---

### Notes on This Round (fill in as you iterate)

| Round | Problem Identified | Decision Rule Added |
|-------|-------------------|---------------------|
| 1     |                   |                     |
| 2     |                   |                     |
| 3     |                   |                     |